# Workstream 4: Issue [L6] — UPOS Bridge (Matteo)

Notebook for developing, validating, and testing the **UPOS Bridge** (`src/ws4_lens/upos_bridge.py`) for autoregressive decoders (**GPT-2** and **GePpeTto**).

---

### Context & Goal (`PROPOSAL.md` Point 5 & `docs/CONTRACTS.md` Section 10)
To test whether an autoregressive language model commits to the **correct grammatical category** of the next word *before* converging on the exact surface word, we map each lens's top-1 predicted vocabulary token $\hat{w}_{t+1}^{(l)}$ to a Universal Dependencies Part-of-Speech (**UPOS**) tag, and compare it with the gold `next_upos`.

### Contract Specifications (`docs/CONTRACTS.md` Section 10 & Issue #24)
1. `build_upos_lexicon(train_table: pd.DataFrame) -> dict[str, str]`:
   - For each exact `form`, find the most frequent UPOS tag.
   - Ties are broken deterministically by the official order in `UPOS_TAGS` (`ADJ, ADP, ADV, AUX, CCONJ, DET, INTJ, NOUN, NUM, PART, PRON, PROPN, PUNCT, SCONJ, SYM, VERB, X`).
   - Also compute lowercase counts and add lowercase keys for words not seen in lowercase, used only as a fallback.
2. `token_to_upos(token: str, lexicon: dict[str, str]) -> str`:
   - Input is the decoded token string: `token = tokenizer.decode([token_id])`.
   - **No leading space**: The token continues the current subword/word, so it does not start a new word $\implies$ return `'X'`.
   - **Otherwise strip space**:
     1. Exact lookup in `lexicon` $\implies$ return tag.
     2. Lowercase lookup in `lexicon` $\implies$ return tag.
     3. Only punctuation characters $\implies$ return `'PUNCT'`.
     4. Only digits (with `.` or `,`) $\implies$ return `'NUM'`.
     5. Else $\implies$ return `'X'`.
3. **Coverage Report**:
   - Share of top-1 predictions at the final layer of `gpt2` on PUD EN that map to a tag other than `'X'`.

## 1. Environment Setup and Imports

In [1]:
import string
import sys
import unicodedata
from pathlib import Path

import pandas as pd
import torch

# Locate repository root and add src/ to sys.path
cwd = Path.cwd().resolve()
ROOT_DIR = cwd if (cwd / "src").exists() else cwd.parent
SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lib.alignment import encode_words
from lib.data import UPOS_TAGS, load_model
from lib.paths import ROOT
from lib.registry import get_model_spec
from lib.word_table import load_word_table, table_to_sentences
from ws4_lens.logit_lens import run_model

print(f"Repository root: {ROOT}")
print(f"UPOS tags ({len(UPOS_TAGS)}): {UPOS_TAGS}")

/Users/matteosalvini/Desktop/Magistrale/HLT/HLT_Project-LLMs_on_Probing_and_Logit_Lens/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repository root: /Users/matteosalvini/Desktop/Magistrale/HLT/HLT_Project-LLMs_on_Probing_and_Logit_Lens
UPOS tags (17): ('ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X')


## 2. Implementation of `build_upos_lexicon` and `token_to_upos`

In [2]:
def build_upos_lexicon(train_table: pd.DataFrame) -> dict[str, str]:
    """Build a word-form to most-frequent UPOS tag mapping from a training word table.

    Parameters
    ----------
    train_table : pd.DataFrame
        Training word table containing at least `form` and `upos` columns.

    Returns
    -------
    dict[str, str]
        Dictionary mapping word forms to their most frequent UPOS tag.
        Ties are broken deterministically by the official order in `UPOS_TAGS`.
        Also includes lowercase forms for words not already in the lexicon.
    """
    if "form" not in train_table.columns or "upos" not in train_table.columns:
        raise ValueError("train_table must contain 'form' and 'upos' columns.")

    tag_rank = {tag: i for i, tag in enumerate(UPOS_TAGS)}

    # Filter out null / NaN rows
    valid = train_table.dropna(subset=["form", "upos"])

    # 1. Exact form counts: count occurrences of each (form, upos)
    exact_counts: dict[str, dict[str, int]] = {}
    lower_counts: dict[str, dict[str, int]] = {}

    for form, upos in zip(valid["form"], valid["upos"]):
        form_str = str(form)
        upos_str = str(upos)

        # Exact
        c_map = exact_counts.setdefault(form_str, {})
        c_map[upos_str] = c_map.get(upos_str, 0) + 1

        # Lowercase
        low_str = form_str.lower()
        l_map = lower_counts.setdefault(low_str, {})
        l_map[upos_str] = l_map.get(upos_str, 0) + 1

    def pick_best_tag(counts: dict[str, int]) -> str:
        # Sort by: -count (descending), then tag_rank (ascending)
        return min(counts.keys(), key=lambda t: (-counts[t], tag_rank.get(t, 999)))

    lexicon: dict[str, str] = {}
    for form_str, counts in exact_counts.items():
        lexicon[form_str] = pick_best_tag(counts)

    # 2. Add lowercase fallback keys for any lower_str not already present
    for low_str, counts in lower_counts.items():
        if low_str not in lexicon:
            lexicon[low_str] = pick_best_tag(counts)

    return lexicon


def _is_punctuation(s: str) -> bool:
    """Check if string consists solely of punctuation characters."""
    if not s:
        return False
    return all(unicodedata.category(c).startswith("P") or c in string.punctuation for c in s)


def _is_number(s: str) -> bool:
    """Check if string consists solely of digits with optional '.' or ',' (must have >= 1 digit)."""
    if not s:
        return False
    has_digit = False
    for c in s:
        if c.isdigit():
            has_digit = True
        elif c in (".", ","):
            continue
        else:
            return False
    return has_digit


def token_to_upos(token: str, lexicon: dict[str, str]) -> str:
    """Map a decoded token string to its most likely UPOS tag.

    Parameters
    ----------
    token : str
        Decoded token string (e.g. from `tokenizer.decode([token_id])`).
    lexicon : dict[str, str]
        Word-form to UPOS lexicon from `build_upos_lexicon`.

    Returns
    -------
    str
        Predicted UPOS tag (one of `UPOS_TAGS`), or `'X'` if subword / unknown.
    """
    # Rule 1: No leading space -> token continues the current word (subword) -> return X
    if not token.startswith(" "):
        return "X"

    # Rule 2: Strip space
    word = token.strip()
    if not word:
        return "X"

    # Rule 3: Exact lookup
    if word in lexicon:
        return lexicon[word]

    # Rule 4: Lowercase lookup
    word_lower = word.lower()
    if word_lower in lexicon:
        return lexicon[word_lower]

    # Rule 5: Only punctuation characters -> PUNCT
    if _is_punctuation(word):
        return "PUNCT"

    # Rule 6: Only digits (with . or ,) -> NUM
    if _is_number(word):
        return "NUM"

    # Rule 7: Else -> X
    return "X"

## 3. Unit Verification on Acceptance Criteria Examples

Verifies all acceptance criteria on a toy table:
1. `" the"` $\implies$ `DET`
2. `"ing"` $\implies$ `X` (no leading space)
3. `" ,"` $\implies$ `PUNCT`
4. `" 1999"` $\implies$ `NUM`
5. Deterministic tie-breaking by `UPOS_TAGS` order
6. Fallback lowercase lookup (e.g. `" Apple"` $\implies$ `PROPN` allows `" apple"` $\implies$ `PROPN`)

In [3]:
# Construct a toy word table
toy_df = pd.DataFrame({
    "form": ["the", "The", "cat", "sat", "on", "the", "mat", ",", "1999", "tie_word", "tie_word"],
    "upos": ["DET", "DET", "NOUN", "VERB", "ADP", "DET", "NOUN", "PUNCT", "NUM", "NOUN", "ADJ"],
})

toy_lexicon = build_upos_lexicon(toy_df)

# 1. Test ' the' -> DET
assert token_to_upos(" the", toy_lexicon) == "DET", f"Expected DET, got {token_to_upos(' the', toy_lexicon)}"
print("✅ Test 1 PASSED: ' the' -> DET")

# 2. Test 'ing' -> X (no leading space)
assert token_to_upos("ing", toy_lexicon) == "X", f"Expected X, got {token_to_upos('ing', toy_lexicon)}"
print("✅ Test 2 PASSED: 'ing' -> X (subword without leading space)")

# 3. Test ' ,' -> PUNCT
assert token_to_upos(" ,", toy_lexicon) == "PUNCT", f"Expected PUNCT, got {token_to_upos(' ,', toy_lexicon)}"
# Also test punctuation not in lexicon
assert token_to_upos(" ;", toy_lexicon) == "PUNCT"
assert token_to_upos(" --", toy_lexicon) == "PUNCT"
print("✅ Test 3 PASSED: ' ,' -> PUNCT (and unseen punctuation -> PUNCT)")

# 4. Test ' 1999' -> NUM
assert token_to_upos(" 1999", toy_lexicon) == "NUM", f"Expected NUM, got {token_to_upos(' 1999', toy_lexicon)}"
# Also test unseen number with dot/comma
assert token_to_upos(" 2026", toy_lexicon) == "NUM"
assert token_to_upos(" 3.14", toy_lexicon) == "NUM"
assert token_to_upos(" 1,000", toy_lexicon) == "NUM"
print("✅ Test 4 PASSED: ' 1999' -> NUM (and unseen numbers with ./, -> NUM)")

# 5. Test deterministic tie breaking: 'tie_word' has 1 NOUN and 1 ADJ. In UPOS_TAGS, ADJ is rank 0, NOUN is rank 7 -> ADJ wins!
assert toy_lexicon["tie_word"] == "ADJ", f"Expected ADJ for tie_word, got {toy_lexicon['tie_word']}"
assert token_to_upos(" tie_word", toy_lexicon) == "ADJ"
print("✅ Test 5 PASSED: Deterministic tie-breaking prefers earlier UPOS_TAGS rank (ADJ over NOUN)")

# 6. Test lowercase fallback: 'London' only seen capitalized
cap_df = pd.DataFrame({"form": ["London"], "upos": ["PROPN"]})
cap_lexicon = build_upos_lexicon(cap_df)
assert cap_lexicon["London"] == "PROPN"
assert cap_lexicon["london"] == "PROPN"  # lowercase fallback added
assert token_to_upos(" london", cap_lexicon) == "PROPN"
assert token_to_upos(" LONDON", cap_lexicon) == "PROPN"
print("✅ Test 6 PASSED: Lowercase fallback mapping works seamlessly")

✅ Test 1 PASSED: ' the' -> DET
✅ Test 2 PASSED: 'ing' -> X (subword without leading space)
✅ Test 3 PASSED: ' ,' -> PUNCT (and unseen punctuation -> PUNCT)
✅ Test 4 PASSED: ' 1999' -> NUM (and unseen numbers with ./, -> NUM)
✅ Test 5 PASSED: Deterministic tie-breaking prefers earlier UPOS_TAGS rank (ADJ over NOUN)
✅ Test 6 PASSED: Lowercase fallback mapping works seamlessly


## 4. Real Treebank Lexicons: EWT (EN) and ISDT (IT)

Build full lexicons from `ewt_train` and `isdt_train` and inspect vocabulary coverage.

In [4]:
# 1. ISDT train lexicon (Italian)
isdt_train = load_word_table("isdt", "train")
isdt_lexicon = build_upos_lexicon(isdt_train)
print(f"ISDT Train Lexicon: {len(isdt_lexicon):,} entries (from {len(isdt_train):,} words)")

# 2. EWT train lexicon (English)
ewt_train = load_word_table("ewt", "train")
ewt_lexicon = build_upos_lexicon(ewt_train)
print(f"EWT Train Lexicon:  {len(ewt_lexicon):,} entries (from {len(ewt_train):,} words)")

ISDT Train Lexicon: 33,216 entries (from 276,014 words)
EWT Train Lexicon:  23,282 entries (from 204,578 words)


## 5. Acceptance Criterion: Coverage Report on PUD EN with GPT-2

Compute the share of top-1 predictions at the **final layer of GPT-2 on PUD EN** that map to a UPOS tag other than `'X'`.

Per `CONTRACTS.md` Section 8:
- Sentences are encoded with `encode_words(tokenizer, sentences, add_bos=True)`.
- For each word $w$ with non-null `next_upos`, the prediction position is the **last subword of $w$**.
- Model top-1 prediction is $\hat{w}_{t+1} = \text{tokenizer.decode}([\text{logits}[t].\text{argmax}()])$.
- Predicted UPOS tag is $\hat{y}_{t+1} = \text{token_to_upos}(\hat{w}_{t+1}, \text{lexicon})$.
- **Coverage** is the proportion of positions where $\hat{y}_{t+1} \neq \text{'X'}$.
- Also report the **Top-1 UPOS Accuracy** (where $\hat{y}_{t+1} == y_{t+1}^\text{gold}$).

In [5]:
# 1. Load GPT-2
spec_gpt2 = get_model_spec("gpt2")
model_gpt2, tokenizer_gpt2, meta_gpt2 = load_model(
    checkpoint=spec_gpt2["hf_id"],
    kind=spec_gpt2["kind"],
    tokenizer_kwargs={"add_prefix_space": True},
)
model_gpt2.eval()
print(f"Loaded GPT-2 on device: {meta_gpt2['device']}")

# 2. Load PUD EN word table
pud_en_table = load_word_table("pud_en", "test")
pud_en_sentences = table_to_sentences(pud_en_table)
print(f"Loaded PUD EN: {len(pud_en_sentences)} sentences, {len(pud_en_table):,} words")

# Group word table by sentence to access gold next_upos easily
sent_groups = dict(tuple(pud_en_table.groupby("sent_id", sort=False)))

total_eval_positions = 0
covered_positions = 0
correct_positions = 0
tag_counter: dict[str, int] = {}

batch_size = 16
for i in range(0, len(pud_en_sentences), batch_size):
    chunk = pud_en_sentences[i : i + batch_size]
    sent_ids = [s_id for s_id, _ in chunk]
    batch_words = [words for _, words in chunk]

    encoded, word_ids_batch = encode_words(
        tokenizer_gpt2,
        batch_words,
        add_bos=True,
        max_length=512,
    )

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    with torch.no_grad():
        outputs = model_gpt2(
            input_ids=input_ids.to(meta_gpt2["device"]),
            attention_mask=attention_mask.to(meta_gpt2["device"]),
        )
        final_logits = outputs.logits  # [B, T, V]

    for b_idx, s_id in enumerate(sent_ids):
        df_sent = sent_groups[s_id]
        word_ids = word_ids_batch[b_idx]
        n_words = len(df_sent)

        # Find the last subword position for each word 0 .. n_words - 1
        # Note: Position 0 is BOS (word_id is None)
        last_subword_pos: dict[int, int] = {}
        for pos, w_id in enumerate(word_ids):
            if w_id is not None and w_id < n_words:
                last_subword_pos[w_id] = pos

        next_upos_list = df_sent["next_upos"].tolist()

        for w_idx in range(n_words):
            gold_next = next_upos_list[w_idx]
            # Skip words without a next_upos (last word of sentence or NaN)
            if pd.isna(gold_next) or gold_next is None:
                continue
            if w_idx not in last_subword_pos:
                continue

            pos_t = last_subword_pos[w_idx]
            pred_id = int(final_logits[b_idx, pos_t, :].argmax().item())
            pred_token = tokenizer_gpt2.decode([pred_id])

            pred_upos = token_to_upos(pred_token, ewt_lexicon)

            total_eval_positions += 1
            tag_counter[pred_upos] = tag_counter.get(pred_upos, 0) + 1
            if pred_upos != "X":
                covered_positions += 1
                if pred_upos == gold_next:
                    correct_positions += 1

cov_rate = (covered_positions / total_eval_positions) * 100.0
acc_rate = (correct_positions / total_eval_positions) * 100.0
cov_acc = (correct_positions / covered_positions * 100.0) if covered_positions > 0 else 0.0

print("\n=======================================================")
print("  GPT-2 Final Layer Coverage Report on PUD EN")
print("=======================================================")
print(f"Total Next-Token Prediction Positions: {total_eval_positions:,}")
print(f"Predictions Mapped to UPOS (Non-'X'): {covered_positions:,} ({cov_rate:.2f}%)")
print(f"Predictions Mapped to 'X' (Subword/Unknown): {total_eval_positions - covered_positions:,} ({100.0 - cov_rate:.2f}%)")
print(f"Top-1 UPOS Accuracy (Overall):         {correct_positions:,} ({acc_rate:.2f}%)")
print(f"Top-1 UPOS Accuracy (on covered):      {correct_positions:,} ({cov_acc:.2f}%)")
print("=======================================================")
print("\nPredicted UPOS Tag Distribution:")
for tag, count in sorted(tag_counter.items(), key=lambda x: -x[1]):
    print(f"  {tag:<6} : {count:>5} ({count / total_eval_positions * 100:>5.2f}%)")

W1118 15:24:16.555000 33482 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
[transformers] The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10633.04it/s]


Loaded GPT-2 on device: mps
Loaded PUD EN: 1000 sentences, 21,180 words

  GPT-2 Final Layer Coverage Report on PUD EN
Total Next-Token Prediction Positions: 20,180
Predictions Mapped to UPOS (Non-'X'): 16,444 (81.49%)
Predictions Mapped to 'X' (Subword/Unknown): 3,736 (18.51%)
Top-1 UPOS Accuracy (Overall):         8,018 (39.73%)
Top-1 UPOS Accuracy (on covered):      8,018 (48.76%)

Predicted UPOS Tag Distribution:
  X      :  3736 (18.51%)
  DET    :  3491 (17.30%)
  NOUN   :  2885 (14.30%)
  ADP    :  2459 (12.19%)
  AUX    :  1426 ( 7.07%)
  ADJ    :  1109 ( 5.50%)
  VERB   :  1050 ( 5.20%)
  PRON   :   809 ( 4.01%)
  PART   :   762 ( 3.78%)
  CCONJ  :   680 ( 3.37%)
  PROPN  :   624 ( 3.09%)
  PUNCT  :   422 ( 2.09%)
  ADV    :   241 ( 1.19%)
  SCONJ  :   225 ( 1.11%)
  NUM    :   215 ( 1.07%)
  SYM    :    46 ( 0.23%)
